# PhysREVE — Leadfield Physics Visualization

This notebook visualises **why the leadfield attention bias helps** by comparing:

- **REVE** — standard self-attention, no physics knowledge
- **PhysREVE** — same attention + $\alpha B$ where $B = L_{\text{row}}\,L_{\text{row}}^\top$

The forward model of EEG is:
$$\mathbf{y} = L \cdot \mathbf{s} + \varepsilon$$

| Symbol | Shape | Meaning |
|---|---|---|
| $\mathbf{y}$ | $(C \times T)$ | Observed EEG at $C$ electrodes |
| $L$ | $(C \times N)$ | Leadfield — maps $N$ brain sources to electrodes |
| $\mathbf{s}$ | $(N \times T)$ | Latent cortical source activations |

The PhysREVE attention bias is:
$$B_{ij} = (L_{\text{row}})_i \cdot (L_{\text{row}})_j = \cos(\ell_i,\, \ell_j) \in [-1, 1]$$

$B_{ij} \approx 1$ means electrodes $i$ and $j$ see the same cortical sources — the transformer should attend strongly between them.

In [1]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

Colab environment ready.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from physreve.physics import build_leadfield, motor_roi_indices
from physreve.datasets.bciv2a import CH_NAMES, SFREQ

N_CH = len(CH_NAMES)
print(f'Channels: {N_CH}  ({CH_NAMES[:5]} ...)')

Channels: 22  (['Fz', 'FC3', 'FC1', 'FCz', 'FC2'] ...)


## 1. Build Leadfield

Uses MNE's 3-shell sphere conductor model — no individual MRI required.
The sphere is fitted automatically to the electrode positions.

- **Shell 1 — Brain:** innermost, cortical sources live here
- **Shell 2 — Skull:** high-resistance barrier, smears source signals
- **Shell 3 — Scalp:** where electrodes sit

We also get back the **raw leadfield** $L$ for visualisation.

In [3]:
L_col_np, L_row_np, src_pos, info, L_raw = build_leadfield(
    CH_NAMES, sfreq=SFREQ, verbose=True, return_raw=True
)

# Attention bias matrix  B = L_row @ L_row^T
B = L_row_np @ L_row_np.T   # (n_ch, n_ch) cosine similarity

# Motor ROI source indices
lh_idx_np, rh_idx_np = motor_roi_indices(info, src_pos, CH_NAMES)

# Electrode positions in metres
elec_np = np.array([info['chs'][i]['loc'][:3] for i in range(N_CH)])

print(f'\nLeadfield L        : {L_raw.shape}  (n_ch × n_src)')
print(f'Attention bias B   : {B.shape}  (n_ch × n_ch)')
print(f'Source positions   : {src_pos.shape}')
print(f'LH motor sources   : {len(lh_idx_np)}')
print(f'RH motor sources   : {len(rh_idx_np)}')

  Source space: 1496 active dipoles
  Leadfield shape: (22, 1496)

Leadfield L        : (22, 1496)  (n_ch × n_src)
Attention bias B   : (22, 22)  (n_ch × n_ch)
Source positions   : (1496, 3)
LH motor sources   : 84
RH motor sources   : 83


## 2. Visualization

Three panels:
1. **3-Shell sphere model** — electrodes on scalp, sources in brain, leadfield projections shown as coloured lines from selected motor sources to the electrodes they activate most strongly
2. **REVE attention bias** — zero (all electrode pairs treated equally, no physics)
3. **PhysREVE attention bias** — $B = L_{\text{row}}\,L_{\text{row}}^\top$, physics-structured

In [4]:
# ── Helper: draw a transparent sphere ────────────────────────────────────────
def _draw_sphere(ax, r, color, alpha, n=28):
    u, v = np.mgrid[0:2*np.pi:n*1j, 0:np.pi:n*1j]
    x = r * np.cos(u) * np.sin(v)
    y = r * np.sin(u) * np.sin(v)
    z = r * np.cos(v)
    ax.plot_surface(x, y, z, color=color, alpha=alpha,
                    linewidth=0, antialiased=True, shade=True)

# ── Figure ────────────────────────────────────────────────────────────────────
BG = '#0d1117'
fig = plt.figure(figsize=(22, 8))
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.14, left=0.04, right=0.97)

ax1 = fig.add_subplot(gs[0], projection='3d')
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

for ax in [ax2, ax3]:
    ax.set_facecolor(BG)

# ── Panel 1: 3-Shell sphere + leadfield projections ──────────────────────────
ax1.set_facecolor(BG)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
for spine in [ax1.xaxis, ax1.yaxis, ax1.zaxis]:
    spine.pane.set_edgecolor('#222')

ax1.set_title('① 3-Shell Head Model\nLeadfield projections (source → electrode)',
               color='white', fontsize=11, pad=8)

# Three shells (scalp, skull, brain)
_draw_sphere(ax1, 0.090, '#f4a460', 0.05)   # scalp
_draw_sphere(ax1, 0.085, '#aaaaaa', 0.09)   # skull
_draw_sphere(ax1, 0.079, '#4a90d9', 0.15)   # brain

# All source dipoles (faint cloud)
ax1.scatter(src_pos[:,0], src_pos[:,1], src_pos[:,2],
            c='#7eb8f7', s=2, alpha=0.20, linewidths=0)

# EEG electrodes
ax1.scatter(elec_np[:,0], elec_np[:,1], elec_np[:,2],
            c='#ff4444', s=50, zorder=6, linewidths=0, label='Electrodes')

# Label a few key electrodes
for name in ['C3', 'C4', 'Cz', 'Fz']:
    if name in CH_NAMES:
        i = CH_NAMES.index(name)
        ax1.text(elec_np[i,0]*1.18, elec_np[i,1]*1.18, elec_np[i,2]*1.18,
                 name, color='#ffaaaa', fontsize=7)

# Motor ROI sources + leadfield lines
for roi_idx, clr, lbl in [
    (lh_idx_np, '#ff8c00', 'LH motor (C3)'),
    (rh_idx_np, '#39ff14', 'RH motor (C4)'),
]:
    picks = roi_idx[:3]
    ax1.scatter(src_pos[picks,0], src_pos[picks,1], src_pos[picks,2],
                c=clr, s=90, zorder=7, label=lbl, edgecolors='white', linewidths=0.5)
    for si in picks:
        col     = L_col_np[:, si]
        col_abs = np.abs(col)
        thresh  = col_abs.max() * 0.22
        strong  = np.where(col_abs > thresh)[0]
        if len(strong) == 0:
            continue
        norms   = col_abs[strong] / col_abs[strong].max()
        for ei, nv in zip(strong, norms):
            ax1.plot([src_pos[si,0], elec_np[ei,0]],
                     [src_pos[si,1], elec_np[ei,1]],
                     [src_pos[si,2], elec_np[ei,2]],
                     c=clr, alpha=float(nv) * 0.70, lw=1.3)

ax1.set_xlim(-0.10, 0.10); ax1.set_ylim(-0.10, 0.10); ax1.set_zlim(-0.10, 0.10)
ax1.tick_params(colors='#666', labelsize=6)
ax1.set_xlabel('x (m)', color='#888', fontsize=8, labelpad=2)
ax1.set_ylabel('y (m)', color='#888', fontsize=8, labelpad=2)
ax1.set_zlabel('z (m)', color='#888', fontsize=8, labelpad=2)

leg = ax1.legend(fontsize=7.5, loc='upper left',
                  facecolor='#1a1a2e', labelcolor='white', framealpha=0.75)

# Shell labels
for txt, clr, y in [('Scalp','#f4a460',0.77),('Skull','#aaaaaa',0.71),('Brain','#7eb8f7',0.65)]:
    ax1.text2D(0.70, y, txt, transform=ax1.transAxes, color=clr, fontsize=9,
               fontweight='bold')

# ── Panel 2: REVE — no physics bias ──────────────────────────────────────────
REVE_bias = np.zeros((N_CH, N_CH))
im2 = ax2.imshow(REVE_bias, aspect='equal', cmap='RdBu_r', vmin=-1, vmax=1,
                  interpolation='nearest')
ax2.set_title('② REVE\nAttention Bias', color='white', fontsize=12,
               fontweight='bold', pad=10)
ax2.set_xlabel('Electrode j', color='#aaa', fontsize=10)
ax2.set_ylabel('Electrode i', color='#aaa', fontsize=10)
ax2.set_xticks(range(N_CH))
ax2.set_xticklabels(CH_NAMES, fontsize=5.5, rotation=90, color='#ccc')
ax2.set_yticks(range(N_CH))
ax2.set_yticklabels(CH_NAMES, fontsize=5.5, color='#ccc')
ax2.tick_params(colors='#444')
for sp in ax2.spines.values(): sp.set_edgecolor('#333')

cb2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cb2.ax.yaxis.set_tick_params(color='#aaa')
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='#aaa')

ax2.text(0.5, -0.18,
    'No physics — all electrode pairs treated equally\n'
    r'$B_{ij}^{\mathrm{REVE}} = 0$ for all $i \neq j$',
    ha='center', transform=ax2.transAxes, fontsize=9,
    color='#8899aa', style='italic')

# ── Panel 3: PhysREVE — leadfield attention bias ──────────────────────────────
im3 = ax3.imshow(B, aspect='equal', cmap='RdBu_r', vmin=-1, vmax=1,
                  interpolation='nearest')
ax3.set_title(
    '③ PhysREVE\n'
    r'Attention Bias  $B = L_{\mathrm{row}}\,L_{\mathrm{row}}^\top$',
    color='white', fontsize=12, fontweight='bold', pad=10)
ax3.set_xlabel('Electrode j', color='#aaa', fontsize=10)
ax3.set_ylabel('Electrode i', color='#aaa', fontsize=10)
ax3.set_xticks(range(N_CH))
ax3.set_xticklabels(CH_NAMES, fontsize=5.5, rotation=90, color='#ccc')
ax3.set_yticks(range(N_CH))
ax3.set_yticklabels(CH_NAMES, fontsize=5.5, color='#ccc')
ax3.tick_params(colors='#444')
for sp in ax3.spines.values(): sp.set_edgecolor('#333')

# Highlight interesting electrode pairs with golden boxes
for (ni, nj) in [('C3','C4'), ('C3','C1'), ('C4','C2'), ('Cz','CPz'), ('Fz','FCz')]:
    if ni in CH_NAMES and nj in CH_NAMES:
        ii, jj = CH_NAMES.index(ni), CH_NAMES.index(nj)
        ax3.add_patch(plt.Rectangle((jj-0.5, ii-0.5), 1, 1,
                                     fill=False, edgecolor='#ffd700', lw=1.8, zorder=5))

cb3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label='Cosine similarity')
cb3.ax.yaxis.set_tick_params(color='#aaa')
cb3.ax.yaxis.label.set_color('#aaa')
plt.setp(cb3.ax.yaxis.get_ticklabels(), color='#aaa')

ax3.text(0.5, -0.18,
    r'$B_{ij} \approx +1$ → share sources → attend together' + '\n'
    r'$B_{ij} \approx 0$ → independent sources → attend less',
    ha='center', transform=ax3.transAxes, fontsize=9,
    color='#8899aa', style='italic')

# ── Shared title ───────────────────────────────────────────────────────────────
fig.suptitle(
    'PhysREVE vs REVE — Physics-Informed Attention Bias\n'
    'The leadfield shapes which electrodes attend to each other in the transformer',
    color='white', fontsize=13, fontweight='bold', y=1.02
)

plt.savefig('physreve_leadfield_vis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved physreve_leadfield_vis.png')

## 3. How to Read the Plots

### Panel ①: 3-Shell Sphere Model
- **Red dots** = EEG electrodes on the scalp shell
- **Blue cloud** = all brain source dipoles in the cortical volume
- **Orange lines** = how a left-hemisphere motor source projects to electrodes (C3, CP3, etc. light up strongly)
- **Green lines** = same for a right-hemisphere motor source (C4, CP4, etc.)
- The leadfield "smears" each source across several electrodes — adjacent electrodes are correlated

### Panel ②: REVE Attention Bias
- Flat grey — **zero physics knowledge**
- Every electrode pair gets the same prior in attention
- The model must learn electrode relationships from scratch during pretraining

### Panel ③: PhysREVE Attention Bias
- **Red (warm)** = electrodes sharing the same sources → strong prior to attend together
- **Blue (cool)** = electrodes seeing different sources → weaker prior
- **Golden boxes** highlight anatomically meaningful pairs:
  - C3↔C4 (bilateral motor cortex — contralateral during MI)
  - Cz↔CPz (central-parietal — motor + proprioception)
  - Fz↔FCz (frontal — motor planning)

### Why this helps
The attention bias bakes in the EEG forward model before any data is seen.
The transformer does not need to *learn* that C3 and CP3 are correlated — the
leadfield tells it directly. This is especially valuable with limited labelled data,
where learning electrode geometry from scratch is expensive and error-prone.